In [5]:
import numpy as np
from scipy.integrate import nquad
from numpy.polynomial.legendre import leggauss

\begin{equation}
    f1 = \int_{0}^{1}\int_{-\infty}^{\infty}  e^{-y^2} x^2 dydx= \frac{\sqrt{\pi}}{3} 
\end{equation}

\begin{equation}
    f2 = \int_{0}^{\pi}\int_{-\infty}^{\infty}  e^{-|y|} \sin(x) dy dx= 4
\end{equation}

In [6]:
#testing functions 

def f1(x, y):
    return np.exp(-y**2) * x**2

def f2(x, y):
    return np.exp(-np.abs(y)) * np.sin(x)    

In [7]:
# Inner integral: finite interval with Gauss-Legendre quadrature
def integrate_x(y, a, b, n=20):
    # Gauss-Legendre nodes and weights on [-1, 1]
    nodes, weights = leggauss(n)
    # Map nodes from [-1, 1] to [a, b]
    mapped_nodes = 0.5 * (nodes * (b - a) + (b + a))
    mapped_weights = 0.5 * (b - a) * weights
    # Evaluate f(x, y) at the nodes
    values = f2(mapped_nodes, y)
    return np.sum(values * mapped_weights)

# Outer integral: infinite range with nquad
def hybrid_integral(a, b):
    def integrand(y):
        return integrate_x(y, a, b, n=20)
    result, err = nquad(lambda y: integrand(y), [[-np.inf, np.inf]])
    return result, err

# Example of f1 
res, err = hybrid_integral(0, np.pi)
print("Integral result:", res, " ± ", err)


Integral result: 3.999999999999997  ±  2.3370427715721285e-10


\begin{equation}
    f3 = \int_{0}^{1}\int_{0}^{1}\int_{-\infty}^{\infty}\int_{-\infty}^{\infty}  (x+y)e^{-(v_x^2 + v_y^2)} dv_x dv_y dx dy= \pi
\end{equation}

In [8]:
def f3(x,y,vx,vy):
    return (x + y) * np.exp(-(vx**2 + vy**2))

In [9]:
# Finite-range Gauss–Legendre integration over x,y
def integrate_xy(vx, vy, ax, bx, ay, by, n=20):
    # Gauss–Legendre nodes and weights
    nodes, weights = leggauss(n)

    # Map nodes from [-1, 1] to [ax, bx] and [ay, by]
    x_nodes = 0.5 * (nodes * (bx - ax) + (bx + ax))
    y_nodes = 0.5 * (nodes * (by - ay) + (by + ay))
    x_weights = 0.5 * (bx - ax) * weights
    y_weights = 0.5 * (by - ay) * weights

    # Tensor product quadrature
    total = 0.0
    for i, xi in enumerate(x_nodes):
        for j, yj in enumerate(y_nodes):
            total += f3(xi, yj, vx, vy) * x_weights[i] * y_weights[j]
    return total

# Outer integral over infinite vx, vy using nquad
def hybrid_integral(ax, bx, ay, by, n=20):
    def integrand(vx, vy):
        return integrate_xy(vx, vy, ax, bx, ay, by, n=n)

    # nquad over (-∞, ∞) × (-∞, ∞)
    result, err = nquad(integrand, [[-np.inf, np.inf], [-np.inf, np.inf]])
    return result, err

# Example usage
res, err = hybrid_integral(0, 1, 0, 1, n=20)
print("Integral result:", res, " ± ", err)

Integral result: 3.1415926535897762  ±  2.51730878400924e-08
